# Analisis Regresi Logistik Biner
- Harvest Walukow
- 164231104

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

# Membaca dataset
df = pd.read_csv('data.csv')
df.head()

,No,Y,X1,X2,X3,X4
0,1,1,1,0,1,0
1,2,1,1,0,0,0
2,3,0,1,0,0,0
3,4,1,1,0,1,1
4,5,0,1,0,0,0


In [2]:
# Fit model
model = smf.logit("Y ~ X1 + X2 + X3 + X4", data=df).fit()
print(model.summary())

         Current function value: 0.515793
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:                      Y   No. Observations:                   30
Model:                          Logit   Df Residuals:                       25
Method:                           MLE   Df Model:                            4
Date:                Sun, 31 May 2026   Pseudo R-squ.:                  0.2462
Time:                        12:12:55   Log-Likelihood:                -15.474
converged:                      False   LL-Null:                       -20.527
Covariance Type:            nonrobust   LLR p-value:                   0.03867
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.9163      0.837     -1.095      0.273      -2.556       0.724
X1             1.1933      1.027      1.162      0.245      -0.82

C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [3]:
# Odds Ratio dan Confidence Interval 95%
or_df = np.exp(model.params).to_frame(name='Odds Ratio')
conf_int = np.exp(model.conf_int())
or_df['95% CI Lower'] = conf_int[0]
or_df['95% CI Upper'] = conf_int[1]
or_df

C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: overflow encountered in exp
  result = func(self.values, **kwargs)


,Odds Ratio,95% CI Lower,95% CI Upper
Intercept,4.000000e-01,0.077606,2.061704
X1,3.298070e+00,0.440524,24.691636
X2,1.695203e+01,0.605391,474.687569
X3,2.495194e-01,0.020473,3.041115
X4,7.384183e+08,0.000000,inf


### Uji Serentak (Simultan / Omnibus Test)
Uji serentak menggunakan nilai *Likelihood Ratio* untuk menguji apakah seluruh variabel prediktor secara bersama-sama berpengaruh signifikan terhadap variabel dependen.
- $H_0: \beta_1 = \beta_2 = \beta_3 = \beta_4 = 0$
- $H_1:$ minimal salah satu $\beta_j \neq 0$
- Statistik Uji: $G = -2 \ln\left(\frac{L_0}{L_p}\right) = 2(\ln L_p - \ln L_0)$

In [4]:
G = 2 * (model.llf - model.llnull)
df_G = model.df_model
p_val_G = 1 - chi2.cdf(G, df_G)

print(f"Statistik G (Omnibus): {G:.4f}")
print(f"d.f.: {df_G}")
print(f"p-value: {p_val_G:.6f}")

Statistik G (Omnibus): 10.1063
d.f.: 4.0
p-value: 0.038675


Uji Hosmer-Lemeshow digunakan untuk mengetahui apakah model yang diperoleh sudah sesuai dengan data empiris.
- $H_0$: Model fit (tidak ada perbedaan nyata antara hasil observasi dengan prediksi model)
- $H_1$: Model tidak fit

In [5]:
def hosmer_lemeshow(model, g=10):
    pi = model.predict()
    y = model.model.endog
    data = pd.DataFrame({'y': y, 'pi': pi}).sort_values('pi')
    data['group'] = pd.qcut(data['pi'].rank(method='first'), g, labels=False)
    
    obstable = pd.crosstab(data['group'], data['y'])
    for col in [0, 1]:
        if col not in obstable.columns:
            obstable[col] = 0
            
    exptable = pd.DataFrame(index=range(g), columns=[0, 1])
    for i in range(g):
        group_data = data[data['group'] == i]
        n_i = len(group_data)
        pi_bar_i = group_data['pi'].mean()
        exptable.loc[i, 1] = n_i * pi_bar_i
        exptable.loc[i, 0] = n_i * (1 - pi_bar_i)
        
    hl_stat = 0
    for i in range(g):
        hl_stat += ((obstable.loc[i, 0] - exptable.loc[i, 0])**2) / exptable.loc[i, 0]
        hl_stat += ((obstable.loc[i, 1] - exptable.loc[i, 1])**2) / exptable.loc[i, 1]
        
    p_value = 1 - chi2.cdf(hl_stat, g - 2)
    return hl_stat, g - 2, p_value

hl_stat, df_hl, p_val_hl = hosmer_lemeshow(model, g=10)
print(f"HL Chi-Square: {hl_stat:.4f}")
print(f"d.f.: {df_hl}")
print(f"p-value: {p_val_hl:.6f}")

HL Chi-Square: 4.7646
d.f.: 8
p-value: 0.782410


Mengukur seberapa baik model dapat mengklasifikasikan kelas Y=0 dan Y=1 dengan threshold probability 0.5.

In [6]:
pred_probs = model.predict()
preds = (pred_probs >= 0.5).astype(int)

# Matriks Klasifikasi
class_matrix = pd.crosstab(df['Y'], preds, rownames=['Actual (Y)'], colnames=['Predicted (Y)'])
print("Matriks Klasifikasi:")
print(class_matrix)

# Hitung persentase ketepatan klasifikasi
accuracy = (preds == df['Y']).mean() * 100
acc_y0 = class_matrix.loc[0, 0] / class_matrix.loc[0].sum() * 100
acc_y1 = class_matrix.loc[1, 1] / class_matrix.loc[1].sum() * 100

print(f"\nKetepatan Klasifikasi Y=0 (Tidak Prematur): {acc_y0:.2f}%")
print(f"Ketepatan Klasifikasi Y=1 (Prematur): {acc_y1:.2f}%")
print(f"Total Ketepatan Klasifikasi Model: {accuracy:.2f}%")

Matriks Klasifikasi:
Predicted (Y)  0   1
Actual (Y)          
0              7   6
1              3  14

Ketepatan Klasifikasi Y=0 (Tidak Prematur): 53.85%
Ketepatan Klasifikasi Y=1 (Prematur): 82.35%
Total Ketepatan Klasifikasi Model: 70.00%
